
# FairWarn-SHS — GraphSAGE Diagnostics and Edge Ablation

This notebook determines whether GraphSAGE performance is affected by:

1. removing peer edges;
2. using all valid peer edges;
3. using reciprocal peer edges only;
4. evaluating connected students separately;
5. changing a small set of model hyperparameters.

The notebook first screens candidate configurations with one seed. It then reruns
the strongest configurations with five fixed seeds.


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib scipy

In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"
FINAL_SEEDS = [42, 123, 456, 789, 1010]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask_np = (
    nodes["Label_Available"].eq(1) &
    nodes["TARGET_AtRisk"].notna()
).to_numpy()

node_to_index = {
    node_id: index for index, node_id in enumerate(nodes["Node_ID"])
}

excluded = {
    "Node_ID", "Roster_Code", "School_Code", "Class_Code",
    "Label_Available", "TARGET_AtRisk"
}
feature_columns = [column for column in nodes.columns if column not in excluded]
X_raw = nodes[feature_columns].copy()

numeric_columns = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = [
    column for column in X_raw.columns if column not in numeric_columns
]

for column in numeric_columns:
    X_raw[column] = X_raw[column].fillna(X_raw[column].median())

for column in categorical_columns:
    mode = X_raw[column].mode(dropna=True)
    X_raw[column] = X_raw[column].fillna(
        mode.iloc[0] if not mode.empty else "Missing"
    )

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_columns),
    ("categorical", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ), categorical_columns)
])

X = preprocessor.fit_transform(X_raw).astype(np.float32)
y = nodes["TARGET_AtRisk"].fillna(-1).astype(int).to_numpy()

def build_edge_index(edge_mode):
    if edge_mode == "none":
        return torch.empty((2, 0), dtype=torch.long)

    selected = edges.copy()
    if edge_mode == "reciprocal":
        selected = selected[selected["Reciprocal"].eq(1)].copy()
    elif edge_mode != "all":
        raise ValueError(f"Unknown edge mode: {edge_mode}")

    pairs = []
    for _, row in selected.iterrows():
        src = row["Source_Node_ID"]
        dst = row["Target_Node_ID"]
        if src in node_to_index and dst in node_to_index:
            s = node_to_index[src]
            d = node_to_index[dst]
            pairs.extend([(s, d), (d, s)])

    if not pairs:
        return torch.empty((2, 0), dtype=torch.long)

    return torch.tensor(pairs, dtype=torch.long).t().contiguous()

EDGE_INDEX = {
    "none": build_edge_index("none"),
    "all": build_edge_index("all"),
    "reciprocal": build_edge_index("reciprocal"),
}

data = Data(
    x=torch.tensor(X, dtype=torch.float32),
    y=torch.tensor(y, dtype=torch.long)
)

degree_all = torch.bincount(
    EDGE_INDEX["all"][0],
    minlength=len(nodes)
)
connected_mask_np = degree_all.numpy() > 0

print("Nodes:", len(nodes))
print("Labelled nodes:", int(labelled_mask_np.sum()))
print("At-risk labelled nodes:", int((y[labelled_mask_np] == 1).sum()))
print("Not-at-risk labelled nodes:", int((y[labelled_mask_np] == 0).sum()))
print("All undirected edges:", EDGE_INDEX["all"].shape[1] // 2)
print("Reciprocal undirected edges:", EDGE_INDEX["reciprocal"].shape[1] // 2)
print("Connected nodes:", int(connected_mask_np.sum()))
print("Isolated nodes:", int((~connected_mask_np).sum()))
print("Encoded feature dimension:", X.shape[1])


In [ ]:

def make_masks(labels, labelled_mask, seed):
    labelled_indices = np.where(labelled_mask)[0]
    labelled_y = labels[labelled_indices]

    train_val_idx, test_idx = train_test_split(
        labelled_indices,
        test_size=0.20,
        stratify=labelled_y,
        random_state=seed
    )

    train_val_y = labels[train_val_idx]
    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=0.1875,
        stratify=train_val_y,
        random_state=seed
    )

    n = len(labels)
    train_mask = torch.zeros(n, dtype=torch.bool)
    val_mask = torch.zeros(n, dtype=torch.bool)
    test_mask = torch.zeros(n, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    return train_mask, val_mask, test_mask

class DiagnosticGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, layers, dropout):
        super().__init__()
        self.layers = layers
        self.dropout = dropout

        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr="mean")

        if layers == 2:
            self.conv2 = SAGEConv(
                hidden_channels,
                hidden_channels // 2,
                aggr="mean"
            )
            output_size = hidden_channels // 2
        else:
            self.conv2 = None
            output_size = hidden_channels

        self.classifier = torch.nn.Linear(output_size, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        if self.conv2 is not None:
            x = self.conv2(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        return self.classifier(x)

def calculate_metrics(y_true, probability, prediction):
    if len(np.unique(y_true)) < 2:
        return {
            "AUC_ROC": np.nan,
            "AUC_PR": np.nan,
            "Precision_AtRisk": precision_score(
                y_true, prediction, zero_division=0
            ),
            "Recall_AtRisk": recall_score(
                y_true, prediction, zero_division=0
            ),
            "F1_AtRisk": f1_score(
                y_true, prediction, zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_true, prediction
            ),
            "Accuracy": accuracy_score(y_true, prediction),
            "Brier_Score": brier_score_loss(y_true, probability),
        }

    return {
        "AUC_ROC": roc_auc_score(y_true, probability),
        "AUC_PR": average_precision_score(y_true, probability),
        "Precision_AtRisk": precision_score(
            y_true, prediction, zero_division=0
        ),
        "Recall_AtRisk": recall_score(
            y_true, prediction, zero_division=0
        ),
        "F1_AtRisk": f1_score(
            y_true, prediction, zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, prediction
        ),
        "Accuracy": accuracy_score(y_true, prediction),
        "Brier_Score": brier_score_loss(y_true, probability),
    }


In [ ]:

def run_experiment(
    edge_mode,
    hidden_channels,
    layers,
    dropout,
    learning_rate,
    seed,
    max_epochs=300,
    patience=30
):
    set_seed(seed)

    train_mask, val_mask, test_mask = make_masks(
        y, labelled_mask_np, seed
    )

    graph = data.clone()
    graph.edge_index = EDGE_INDEX[edge_mode]
    graph.train_mask = train_mask
    graph.val_mask = val_mask
    graph.test_mask = test_mask
    graph = graph.to(device)

    model = DiagnosticGraphSAGE(
        in_channels=graph.num_node_features,
        hidden_channels=hidden_channels,
        layers=layers,
        dropout=dropout
    ).to(device)

    train_labels = graph.y[graph.train_mask]
    counts = torch.bincount(train_labels, minlength=2).float()
    class_weights = counts.sum() / (2.0 * counts.clamp_min(1.0))
    class_weights = class_weights.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=5e-4
    )

    best_state = None
    best_val_auc_pr = -np.inf
    best_epoch = 0
    wait = 0
    history = []

    started = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(
            logits[graph.train_mask],
            graph.y[graph.train_mask],
            weight=class_weights
        )
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(graph.x, graph.edge_index)
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            val_true = graph.y[graph.val_mask].cpu().numpy()
            val_prob = probabilities[graph.val_mask].cpu().numpy()
            val_auc_pr = average_precision_score(val_true, val_prob)

        history.append({
            "Epoch": epoch,
            "Training_Loss": float(loss.item()),
            "Validation_AUC_PR": float(val_auc_pr)
        })

        if val_auc_pr > best_val_auc_pr + 1e-6:
            best_val_auc_pr = val_auc_pr
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    runtime = time.perf_counter() - started
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = torch.argmax(logits, dim=1)

    test_indices = torch.where(graph.test_mask)[0].cpu().numpy()
    truth = graph.y[graph.test_mask].cpu().numpy()
    probability = probabilities[graph.test_mask].cpu().numpy()
    prediction = predictions[graph.test_mask].cpu().numpy()

    overall = calculate_metrics(truth, probability, prediction)

    connected_test = connected_mask_np[test_indices]
    connected_metrics = {}
    if connected_test.sum() > 1:
        connected_metrics = calculate_metrics(
            truth[connected_test],
            probability[connected_test],
            prediction[connected_test]
        )

    result = {
        "Edge_Mode": edge_mode,
        "Hidden_Channels": hidden_channels,
        "Layers": layers,
        "Dropout": dropout,
        "Learning_Rate": learning_rate,
        "Seed": seed,
        "Best_Epoch": best_epoch,
        "Validation_AUC_PR": best_val_auc_pr,
        "Runtime_Seconds": runtime,
        **overall
    }

    for key, value in connected_metrics.items():
        result[f"Connected_{key}"] = value

    predictions_table = pd.DataFrame({
        "Node_Index": test_indices,
        "Node_ID": nodes.iloc[test_indices]["Node_ID"].to_numpy(),
        "True_Label": truth,
        "Predicted_Label": prediction,
        "AtRisk_Probability": probability,
        "Connected": connected_test.astype(int),
        "Edge_Mode": edge_mode,
        "Seed": seed
    })

    return result, pd.DataFrame(history), predictions_table



## Stage 1: Fast configuration screening

This stage uses seed 42 to screen a small, predefined set of configurations.
It is not the final reported result. The strongest configurations are rerun with
five seeds in Stage 2.


In [ ]:

candidate_configs = []

for edge_mode in ["none", "all", "reciprocal"]:
    for hidden_channels in [32, 64]:
        for layers in [1, 2]:
            for dropout in [0.20, 0.35]:
                for learning_rate in [0.001, 0.005]:
                    candidate_configs.append({
                        "edge_mode": edge_mode,
                        "hidden_channels": hidden_channels,
                        "layers": layers,
                        "dropout": dropout,
                        "learning_rate": learning_rate
                    })

print("Screening configurations:", len(candidate_configs))


In [ ]:

screening_rows = []

for number, config in enumerate(candidate_configs, start=1):
    result, _, _ = run_experiment(
        **config,
        seed=42,
        max_epochs=250,
        patience=25
    )
    screening_rows.append(result)

    print(
        f"{number:02d}/{len(candidate_configs)} | "
        f"{config['edge_mode']:10s} | "
        f"H={config['hidden_channels']} | "
        f"L={config['layers']} | "
        f"D={config['dropout']} | "
        f"LR={config['learning_rate']} | "
        f"AUC-PR={result['AUC_PR']:.4f}"
    )

screening_df = pd.DataFrame(screening_rows)
screening_df = screening_df.sort_values(
    "AUC_PR", ascending=False
).reset_index(drop=True)

screening_df.head(12)


In [ ]:

# Select the best configuration from each edge mode.
selected_configs = []

for edge_mode in ["none", "all", "reciprocal"]:
    best = (
        screening_df[screening_df["Edge_Mode"].eq(edge_mode)]
        .sort_values("AUC_PR", ascending=False)
        .iloc[0]
    )

    selected_configs.append({
        "edge_mode": best["Edge_Mode"],
        "hidden_channels": int(best["Hidden_Channels"]),
        "layers": int(best["Layers"]),
        "dropout": float(best["Dropout"]),
        "learning_rate": float(best["Learning_Rate"])
    })

pd.DataFrame(selected_configs)



## Stage 2: Five-seed confirmation

The best configuration from each edge condition is rerun using five fixed seeds.
These results are suitable for mean ± standard deviation reporting.


In [ ]:

final_rows = []
final_predictions = []
final_histories = []

for config in selected_configs:
    for seed in FINAL_SEEDS:
        result, history, predictions = run_experiment(
            **config,
            seed=seed,
            max_epochs=500,
            patience=40
        )

        final_rows.append(result)

        history["Edge_Mode"] = config["edge_mode"]
        history["Seed"] = seed
        final_histories.append(history)
        final_predictions.append(predictions)

        print(
            f"{config['edge_mode']:10s} | Seed {seed} | "
            f"AUC-PR={result['AUC_PR']:.4f} | "
            f"Recall={result['Recall_AtRisk']:.4f} | "
            f"F1={result['F1_AtRisk']:.4f}"
        )

final_metrics_df = pd.DataFrame(final_rows)
final_history_df = pd.concat(final_histories, ignore_index=True)
final_predictions_df = pd.concat(final_predictions, ignore_index=True)


In [ ]:

metric_columns = [
    "AUC_ROC",
    "AUC_PR",
    "Precision_AtRisk",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Balanced_Accuracy",
    "Accuracy",
    "Brier_Score",
    "Connected_AUC_ROC",
    "Connected_AUC_PR",
    "Connected_Precision_AtRisk",
    "Connected_Recall_AtRisk",
    "Connected_F1_AtRisk",
    "Connected_Balanced_Accuracy"
]

summary_rows = []

for edge_mode, group in final_metrics_df.groupby("Edge_Mode"):
    row = {"Edge_Mode": edge_mode}

    config_columns = [
        "Hidden_Channels", "Layers", "Dropout", "Learning_Rate"
    ]
    for column in config_columns:
        row[column] = group[column].iloc[0]

    for metric in metric_columns:
        if metric in group.columns:
            row[f"{metric}_Mean"] = group[metric].mean()
            row[f"{metric}_SD"] = group[metric].std(ddof=1)

    summary_rows.append(row)

diagnostic_summary_df = pd.DataFrame(summary_rows)
diagnostic_summary_df = diagnostic_summary_df.sort_values(
    "AUC_PR_Mean", ascending=False
).reset_index(drop=True)

diagnostic_summary_df


In [ ]:

plt.figure(figsize=(8, 5))

labels = diagnostic_summary_df["Edge_Mode"]
means = diagnostic_summary_df["AUC_PR_Mean"]
errors = diagnostic_summary_df["AUC_PR_SD"]

plt.bar(labels, means, yerr=errors, capsize=5)
plt.ylabel("Test AUC-PR")
plt.xlabel("Edge condition")
plt.title("GraphSAGE edge ablation across five seeds")
plt.tight_layout()
plt.show()


In [ ]:

screening_df.to_csv(
    "graphsage_diagnostic_screening.csv",
    index=False
)
final_metrics_df.to_csv(
    "graphsage_diagnostic_seed_metrics.csv",
    index=False
)
diagnostic_summary_df.to_csv(
    "graphsage_diagnostic_summary_mean_sd.csv",
    index=False
)
final_predictions_df.to_csv(
    "graphsage_diagnostic_predictions.csv",
    index=False
)
final_history_df.to_csv(
    "graphsage_diagnostic_training_history.csv",
    index=False
)

files.download("graphsage_diagnostic_screening.csv")
files.download("graphsage_diagnostic_seed_metrics.csv")
files.download("graphsage_diagnostic_summary_mean_sd.csv")
files.download("graphsage_diagnostic_predictions.csv")
files.download("graphsage_diagnostic_training_history.csv")
